# Experiment 4 — No Theme

Epic description + Epic success criteria + Stage + base L3 candidates. Theme context is intentionally excluded.

## What the LLM sees

The **user prompt** is structured as:

```text
task
epic
  ├─ description
  └─ success_criteria
value_stream_stage
  ├─ stage_id
  ├─ stage_name
  ├─ stage_description
  ├─ entrance_criteria
  └─ exit_criteria
candidate_l3_capabilities[]
  ├─ capability_id
  ├─ capability_name
  ├─ capability_description
  └─ capability_tier
selection_instruction
```

**Not sent to the LLM:** Theme context, L1/L2 hierarchy, ground truth.


## Configuration and imports

In [ ]:
from pathlib import Path
from time import perf_counter
import ast
import io
import json
import os
import re
import tokenize

import httpx
import pandas as pd
from IPython.display import display

from common import (
    call_llm_with_metrics,
    load_gateway,
    parse_json_response,
    save_results_excel,
    score_sets,
    validate_l3_response,
)

NOTEBOOK_DIR = Path.cwd()
WORKSPACE_DIR = (
    NOTEBOOK_DIR.parent
    if (NOTEBOOK_DIR.parent / "epic_gen.csv").exists()
    else NOTEBOOK_DIR
)
DATA_DIR = Path(os.getenv("L3_EXPERIMENT_DATA_DIR", WORKSPACE_DIR))

THEME_PATH = Path(os.getenv("L3_THEME_PATH", DATA_DIR / "epic_gen.csv"))
STAGE_PATH = Path(os.getenv("L3_STAGE_PATH", DATA_DIR / "VSSrv.csv"))
STAGE_CAPABILITY_MAP_PATH = Path(
    os.getenv(
        "L3_STAGE_CAPABILITY_MAP_PATH",
        DATA_DIR / "VSSCaprv (1).csv",
    )
)
GROUND_TRUTH_PATH = Path(
    os.getenv(
        "L3_GROUND_TRUTH_PATH",
        NOTEBOOK_DIR / "epic_l3_ground_truth_all_themes.xlsx",
    )
)

# Run every Theme in epic_gen.csv. Ground truth is not consulted here.
THEME_IDS = (
    pd.read_csv(
        THEME_PATH,
        usecols=["key"],
        encoding="cp1252",
        encoding_errors="replace",
        dtype=str,
    )["key"]
    .dropna()
    .str.strip()
    .loc[lambda values: values.ne("")]
    .drop_duplicates()
    .tolist()
)

VALUE_STREAM_STAGE_FIELD_ID = "customfield_18700"

# Optional single-example inspection. Leave None for batch execution only.
INSPECTION_THEME_ID = None
INSPECTION_EPIC_KEY = None

EXPERIMENT_NAME = "E4_NO_THEME"


## Retrieval

In [ ]:
def clean_text(v):
    if v is None or (not isinstance(v, (list, dict)) and pd.isna(v)):
        return ""
    return str(v).strip()

def parse_exported_list(v):
    if v is None or pd.isna(v):
        return []
    text = str(v).strip()
    if not text:
        return []
    if text.startswith("[") and text.endswith("]"):
        values = [ast.literal_eval(t.string) for t in tokenize.generate_tokens(io.StringIO(text).readline) if t.type == tokenize.STRING]
        if values:
            return [clean_text(x) for x in values]
    try:
        parsed = ast.literal_eval(text)
    except (SyntaxError, ValueError):
        return [text]
    return [clean_text(x) for x in parsed] if isinstance(parsed, (list, tuple, set)) else [clean_text(parsed)]

def read_table(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, dtype=str, encoding="cp1252", encoding_errors="replace")
    return pd.read_excel(path, dtype=str)

def load_themes():
    frame = read_table(THEME_PATH)
    frame = frame.loc[frame["key"].isin(THEME_IDS)]
    themes = {}

    for _, row in frame.iterrows():
        epic_keys = parse_exported_list(row.get("epic_keys"))
        epic_descriptions = parse_exported_list(row.get("epic_description"))
        epic_success_criteria = parse_exported_list(row.get("epic_successCriteria"))
        epics = [
            {
                "key": epic_key,
                "description": epic_descriptions[index] if index < len(epic_descriptions) else "",
                "success_criteria": epic_success_criteria[index] if index < len(epic_success_criteria) else "",
            }
            for index, epic_key in enumerate(epic_keys)
        ]
        themes[clean_text(row["key"])] = {
            "epics": epics,
        }

    return themes

def jira_headers():
    if os.getenv("JIRA_HEADERS_JSON"):
        return json.loads(os.environ["JIRA_HEADERS_JSON"])
    token = os.getenv("JIRA_BEARER_TOKEN") or os.getenv("JIRA_TOKEN")
    if token:
        return {"Authorization": f"Bearer {token}", "Accept": "application/json"}
    raise RuntimeError("Set JIRA_HEADERS_JSON, JIRA_BEARER_TOKEN, or JIRA_TOKEN.")

def epic_stage_ids(epic_key):
    url = f"{os.environ['JIRA_BASE_URL'].rstrip('/')}/rest/api/2/issue/{epic_key}"
    verify = os.getenv("JIRA_VERIFY_SSL", "false").lower() == "true"
    with httpx.Client(headers=jira_headers(), verify=verify, timeout=30) as client:
        response = client.get(url, params={"fields": VALUE_STREAM_STAGE_FIELD_ID})
        response.raise_for_status()
    raw = response.json().get("fields", {}).get(VALUE_STREAM_STAGE_FIELD_ID) or []
    raw = raw if isinstance(raw, list) else [raw]
    ids = []
    for value in raw:
        text = json.dumps(value, ensure_ascii=False) if isinstance(value, (dict, list)) else clean_text(value)
        ids.extend(re.findall(r"VSS\d+", text, flags=re.IGNORECASE))
    return list(dict.fromkeys(x.upper() for x in ids))

themes = load_themes()
stage_frame = read_table(STAGE_PATH)
stage_capability_map = read_table(STAGE_CAPABILITY_MAP_PATH)

def stage_context(stage_id):
    match = stage_frame.loc[stage_frame["Value Stream Stage ID"].astype(str).str.strip() == stage_id]
    if match.empty:
        raise KeyError(f"No stage metadata for {stage_id}")
    row = match.iloc[0]
    return {"stage_id": stage_id, "stage_name": clean_text(row["Value Stream Stage Name"]), "stage_description": clean_text(row["Value Stream Stage Description"]), "entrance_criteria": clean_text(row["Value Stream Stage Entrance Criteria"]), "exit_criteria": clean_text(row["Value Stream Stage Exit Criteria"])}


## Candidate construction

In [ ]:
def candidate_rows_for_stage(stage_id):
    rows = stage_capability_map.loc[
        stage_capability_map["Value Stream Stage ID"].astype(str).str.strip()
        == stage_id
    ].copy()
    rows = (
        rows.drop_duplicates(subset=["Capability ID"], keep="first")
        .sort_values(["Capability Name", "Capability ID"], kind="stable")
    )

    return [
        {
            "capability_id": clean_text(row["Capability ID"]),
            "capability_name": clean_text(row["Capability Name"]),
            "capability_description": clean_text(row["Capability Description"]),
            "capability_tier": clean_text(row["Capability Tier"]),
        }
        for _, row in rows.iterrows()
    ]


## Production prompt

In [ ]:
SYSTEM_PROMPT = """You are an enterprise Business Capability Architecture specialist performing Level 3 (L3) business capability classification.

OBJECTIVE
Select only candidate L3 capabilities materially represented, enabled, changed, enhanced, or required by the supplied business context. This is capability classification, not keyword matching.

EVIDENCE PRIORITY
Use only fields that are present, in this order when available:
1. Epic success criteria
2. Epic description
3. Value Stream Stage context
4. Theme business needs
5. Theme description

If Epic context is absent, do not assume or refer to it.

CANDIDATE INTERPRETATION
- capability_description is the primary semantic definition.
- capability_name is the supporting label.
- capability_tier is supporting taxonomy context only.
- level_1_name and level_2_name, when supplied, are for disambiguation only and must never independently justify a selection.

DECISION PROCEDURE
1. Determine the core business function or outcome represented by the supplied context.
2. Compare it semantically against every candidate's capability_description.
3. Select a candidate only when direct evidence shows that its business function is materially represented; relatedness alone is insufficient.
4. Use Value Stream Stage context to constrain or disambiguate, but Stage membership alone is not evidence.
5. Use Theme context as broader strategic context; it must not overpower more specific Epic evidence.
6. When candidates overlap, prefer the most specific directly aligned capability.

DO NOT SELECT
Do not select a capability merely because of shared keywords, hierarchy family, Stage membership, upstream/downstream relationship, data exchange, stakeholder involvement, technical adjacency, or general Theme relevance. Do not map technical implementation details unless the business capability itself is explicitly enabled or changed.

MULTI-SELECTION
Select 0 to 3 capabilities. Default to one when one capability adequately represents the function. Select multiple only for distinct material business functions with independent evidence. Return {"l3": []} when none is sufficiently supported.

REASONS
For every selection, give a concise reason connecting supplied evidence to the candidate definition. Use only exact capability_id values from the supplied candidates; never invent or alter an ID.

FINAL VALIDATION
Before responding, verify that every selected ID is a supplied candidate, every selection has direct evidence, no selection is merely adjacent, no stronger or more-specific candidate was omitted, multiple selections are genuinely distinct, and no more than three capabilities are selected.

OUTPUT CONTRACT
Return JSON only, with no Markdown, code fences, commentary, or extra fields:
{"l3":[{"capability_id":"CAP00000000","reason":"Concise evidence-based explanation."}]}"""

def build_user_prompt(theme, epic, stage, candidate_rows):
    payload = {"task": "Select the materially represented L3 business capabilities from the supplied candidates.", "epic": {"description": epic["description"], "success_criteria": epic["success_criteria"]}, "value_stream_stage": stage, "candidate_l3_capabilities": candidate_rows, "selection_instruction": "Select 0 to 3 candidates; return an empty l3 list when none has direct evidence."}
    return json.dumps(payload, ensure_ascii=False, indent=2)


## Prediction

In [ ]:
def predict_for_stage(gateway, theme, epic, stage_id):
    stage = stage_context(stage_id)
    candidates = candidate_rows_for_stage(stage_id)
    if not candidates:
        return {
            "stage": stage,
            "candidates": [],
            "user_prompt": build_user_prompt(theme, epic, stage, []),
            "raw_response": None,
            "selections": [],
            "metrics": None,
        }

    user_prompt = build_user_prompt(theme, epic, stage, candidates)
    raw_response, metrics = call_llm_with_metrics(
        gateway,
        SYSTEM_PROMPT,
        user_prompt,
    )
    selections = validate_l3_response(
        parse_json_response(raw_response),
        [candidate["capability_id"] for candidate in candidates],
        allow_empty=True,
        max_selected=3,
    )
    return {
        "stage": stage,
        "candidates": candidates,
        "user_prompt": user_prompt,
        "raw_response": raw_response,
        "selections": selections,
        "metrics": metrics,
    }


def metric_text(value):
    return "n/a" if value is None else str(value)


def summarize_llm_calls(call_metrics):
    successful = call_metrics.loc[call_metrics["status"] == "ok"].copy()

    def numeric(column):
        return pd.to_numeric(successful[column], errors="coerce").dropna()

    latency = numeric("latency_seconds")
    input_tokens = numeric("input_tokens")
    output_tokens = numeric("output_tokens")
    total_tokens = numeric("total_tokens")

    return pd.DataFrame([{
        "successful_calls": len(successful),
        "failed_calls": int((call_metrics["status"] == "error").sum()),
        "usage_reported_calls": len(total_tokens),
        "avg_latency_seconds": float(latency.mean()) if len(latency) else None,
        "p50_latency_seconds": float(latency.quantile(0.50)) if len(latency) else None,
        "p95_latency_seconds": float(latency.quantile(0.95)) if len(latency) else None,
        "avg_input_tokens": float(input_tokens.mean()) if len(input_tokens) else None,
        "avg_output_tokens": float(output_tokens.mean()) if len(output_tokens) else None,
        "avg_total_tokens": float(total_tokens.mean()) if len(total_tokens) else None,
        "total_input_tokens": int(input_tokens.sum()) if len(input_tokens) else None,
        "total_output_tokens": int(output_tokens.sum()) if len(output_tokens) else None,
        "total_tokens": int(total_tokens.sum()) if len(total_tokens) else None,
    }])


def run_predictions():
    gateway = load_gateway()
    prediction_rows = []
    call_rows = []
    total_epics = sum(len(theme["epics"]) for theme in themes.values())
    epic_index = 0

    print(f"Running {EXPERIMENT_NAME}: {len(themes)} themes / {total_epics} epics")

    for theme_id, theme in themes.items():
        for epic in theme["epics"]:
            epic_index += 1
            epic_key = epic["key"]
            stage_ids = []
            stage_predictions = []
            available_ids = set()
            predicted_ids = set()
            reasons = []
            status = "ok"
            error = None

            print(f"\n[{epic_index}/{total_epics}] {theme_id} | {epic_key}")

            try:
                stage_ids = epic_stage_ids(epic_key)
            except Exception as exc:
                status = "error"
                error = str(exc)
                print(f"  JIRA ERROR | {error}")

            if status == "ok" and not stage_ids:
                status = "no_stage"
                print("  SKIP | no Value Stream Stage")

            if status == "ok":
                for stage_id in stage_ids:
                    started = perf_counter()
                    try:
                        result = predict_for_stage(gateway, theme, epic, stage_id)
                        candidates = result["candidates"]
                        available_ids.update(
                            candidate["capability_id"] for candidate in candidates
                        )

                        if not candidates:
                            print(f"  {stage_id} SKIP | no candidates")
                            continue

                        metrics = result["metrics"]
                        selected_ids = [
                            selection["capability_id"]
                            for selection in result["selections"]
                        ]
                        print(
                            f"  {stage_id} OK"
                            f" | candidates={len(candidates)}"
                            f" | latency={metrics['latency_seconds']:.3f}s"
                            f" | input_tokens={metric_text(metrics['input_tokens'])}"
                            f" | output_tokens={metric_text(metrics['output_tokens'])}"
                            f" | total_tokens={metric_text(metrics['total_tokens'])}"
                            f" | selected={selected_ids}"
                        )

                        call_rows.append({
                            "experiment": EXPERIMENT_NAME,
                            "theme_id": theme_id,
                            "epic_key": epic_key,
                            "stage_id": stage_id,
                            "candidate_count": len(candidates),
                            "status": "ok",
                            "latency_seconds": metrics["latency_seconds"],
                            "input_tokens": metrics["input_tokens"],
                            "output_tokens": metrics["output_tokens"],
                            "total_tokens": metrics["total_tokens"],
                            "selected_count": len(selected_ids),
                            "error": None,
                        })
                        stage_predictions.append({
                            "stage_id": stage_id,
                            "selections": result["selections"],
                        })
                        for selection in result["selections"]:
                            predicted_ids.add(selection["capability_id"])
                            reasons.append({"stage_id": stage_id, **selection})
                    except Exception as exc:
                        latency = perf_counter() - started
                        status = "error"
                        error = str(exc)
                        print(f"  {stage_id} ERROR | latency={latency:.3f}s | {error}")
                        call_rows.append({
                            "experiment": EXPERIMENT_NAME,
                            "theme_id": theme_id,
                            "epic_key": epic_key,
                            "stage_id": stage_id,
                            "candidate_count": None,
                            "status": "error",
                            "latency_seconds": latency,
                            "input_tokens": None,
                            "output_tokens": None,
                            "total_tokens": None,
                            "selected_count": None,
                            "error": error,
                        })
                        break

            if status == "ok" and stage_ids and not available_ids:
                status = "no_candidates"

            prediction_rows.append({
                "experiment": EXPERIMENT_NAME,
                "theme_id": theme_id,
                "epic_key": epic_key,
                "stage_ids": json.dumps(stage_ids),
                "available_candidate_l3_ids": json.dumps(sorted(available_ids)),
                "predicted_l3_ids": json.dumps(sorted(predicted_ids)),
                "model_reasons": json.dumps(reasons, ensure_ascii=False),
                "stage_predictions": json.dumps(stage_predictions, ensure_ascii=False),
                "status": status,
                "error": error,
            })

    call_columns = [
        "experiment",
        "theme_id",
        "epic_key",
        "stage_id",
        "candidate_count",
        "status",
        "latency_seconds",
        "input_tokens",
        "output_tokens",
        "total_tokens",
        "selected_count",
        "error",
    ]
    return (
        pd.DataFrame(prediction_rows),
        pd.DataFrame(call_rows, columns=call_columns),
    )


## Single-example inspection

In [ ]:
if INSPECTION_THEME_ID and INSPECTION_EPIC_KEY:
    theme = themes[INSPECTION_THEME_ID]
    epic = next(
        item
        for item in theme["epics"]
        if item["key"] == INSPECTION_EPIC_KEY
    )
    stage_id = epic_stage_ids(epic["key"])[0]
    stage = stage_context(stage_id)
    candidates = candidate_rows_for_stage(stage_id)
    user_prompt = build_user_prompt(theme, epic, stage, candidates)

    print("SYSTEM PROMPT")
    print(SYSTEM_PROMPT)
    print("\nUSER PROMPT")
    print(user_prompt)
    print("\nCANDIDATES")
    display(pd.DataFrame(candidates))

    if candidates:
        result = predict_for_stage(
            load_gateway(),
            theme,
            epic,
            stage_id,
        )
        print("\nMODEL RESPONSE")
        print(result["raw_response"])
        print("\nCALL METRICS")
        display(pd.DataFrame([result["metrics"]]))
    else:
        print("\nNo candidates for this Stage; LLM call skipped.")
else:
    print(
        "Set INSPECTION_THEME_ID and INSPECTION_EPIC_KEY "
        "to inspect one example."
    )


## Batch execution and evaluation

Predictions are produced before ground truth is loaded.

Only **valid evaluation Epics** are scored. An Epic is valid only when:

1. it has non-empty Jira L3 ground truth;
2. prediction/retrieval completed successfully;
3. it has a Value Stream Stage and candidate L3s; and
4. **every ground-truth L3 is present in the candidate set supplied to the LLM**.

Rows that fail any of these checks are kept only as diagnostics and do not contribute to exact match, precision, recall, or F1.


In [ ]:
def ground_truth_by_epic():
    ground_truth = read_table(GROUND_TRUTH_PATH).copy()
    ground_truth["l3_capability_id"] = (
        ground_truth["l3_capability_id"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    # Blank/no-L3 rows are unlabeled, not negative examples.
    ground_truth = ground_truth.loc[
        ground_truth["l3_capability_id"].ne("")
    ]

    return {
        epic_key: set(group["l3_capability_id"])
        for epic_key, group in ground_truth.groupby(
            "epic_key",
            sort=False,
        )
    }


def empty_metrics(predicted_count, truth_count):
    return {
        "exact_match": None,
        "precision": None,
        "recall": None,
        "f1": None,
        "predicted_count": predicted_count,
        "truth_count": truth_count,
    }


def evaluate_predictions(prediction_frame):
    truth_by_epic = ground_truth_by_epic()
    result_rows = []

    for row in prediction_frame.to_dict(orient="records"):
        predicted_ids = set(json.loads(row["predicted_l3_ids"]))
        available_candidate_ids = set(
            json.loads(row["available_candidate_l3_ids"])
        )
        truth_ids = truth_by_epic.get(row["epic_key"])

        if truth_ids is None:
            available_truth_ids = None
            availability_fraction = None
            exclusion_reason = "missing_ground_truth"
        else:
            available_truth_ids = truth_ids & available_candidate_ids
            availability_fraction = (
                len(available_truth_ids) / len(truth_ids)
            )

            if row["status"] == "error":
                exclusion_reason = "error"
            elif row["status"] == "no_stage":
                exclusion_reason = "no_stage"
            elif row["status"] == "no_candidates" or not available_candidate_ids:
                exclusion_reason = "no_candidates"
            elif not truth_ids.issubset(available_candidate_ids):
                # GT and Stage→L3 candidate mapping are inconsistent.
                # Do not use this Epic to judge the LLM selector.
                exclusion_reason = "gt_not_fully_retrievable"
            else:
                exclusion_reason = ""

        evaluation_eligible = exclusion_reason == ""

        if evaluation_eligible:
            metrics = score_sets(predicted_ids, truth_ids)
        else:
            metrics = empty_metrics(
                predicted_count=len(predicted_ids),
                truth_count=(len(truth_ids) if truth_ids is not None else None),
            )

        row["ground_truth_l3_ids"] = (
            json.dumps(sorted(truth_ids))
            if truth_ids is not None
            else None
        )
        row["gt_available_candidate_l3_ids"] = (
            json.dumps(sorted(available_truth_ids))
            if available_truth_ids is not None
            else None
        )
        row["gt_candidate_available_count"] = (
            len(available_truth_ids)
            if available_truth_ids is not None
            else None
        )
        row["gt_candidate_availability_fraction"] = availability_fraction
        row["evaluation_eligible"] = evaluation_eligible
        row["evaluation_exclusion_reason"] = exclusion_reason
        row.update(metrics)
        result_rows.append(row)

    return pd.DataFrame(result_rows)


def evaluation_summary(results):
    valid = results.loc[results["evaluation_eligible"]].copy()

    summary = pd.DataFrame([
        {
            "scope": "valid_evaluation_population",
            "evaluated_epics": len(valid),
            "exact_match_accuracy": (
                valid["exact_match"].mean() if len(valid) else 0.0
            ),
            "mean_precision": (
                valid["precision"].mean() if len(valid) else 0.0
            ),
            "mean_recall": (
                valid["recall"].mean() if len(valid) else 0.0
            ),
            "mean_f1": (
                valid["f1"].mean() if len(valid) else 0.0
            ),
        }
    ])

    diagnostics = pd.DataFrame([
        {
            "prediction_rows": len(results),
            "valid_evaluation_epics": len(valid),
            "excluded_from_evaluation": int((~results["evaluation_eligible"]).sum()),
            "missing_ground_truth": int(
                (results["evaluation_exclusion_reason"] == "missing_ground_truth").sum()
            ),
            "no_stage": int(
                (results["evaluation_exclusion_reason"] == "no_stage").sum()
            ),
            "no_candidates": int(
                (results["evaluation_exclusion_reason"] == "no_candidates").sum()
            ),
            "gt_not_fully_retrievable": int(
                (
                    results["evaluation_exclusion_reason"]
                    == "gt_not_fully_retrievable"
                ).sum()
            ),
            "errors": int(
                (results["evaluation_exclusion_reason"] == "error").sum()
            ),
        }
    ])

    return summary, diagnostics


predictions, llm_calls = run_predictions()
results = evaluate_predictions(predictions)
summary, diagnostics = evaluation_summary(results)
llm_call_summary = summarize_llm_calls(llm_calls)

print("\nEvaluation summary — valid Epics only")
display(summary)

print("\nExcluded-row diagnostics")
display(diagnostics)

print("\nLLM latency / token summary")
display(llm_call_summary)

print("\nPer-call LLM metrics")
display(llm_calls.head(50))

display(results.head(20))

output_path = save_results_excel(
    results,
    EXPERIMENT_NAME,
    "results",
    extra_sheets={
        "evaluation_summary": summary,
        "diagnostics": diagnostics,
        "llm_calls": llm_calls,
        "llm_call_summary": llm_call_summary,
    },
)
print(f"Saved {output_path}")
